In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import json

# Task 1.1

In [ ]:
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct-AWQ",device_map="cuda",torch_dtype="float16") 
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct-AWQ") 

In [12]:
messages = [{"role": "user", "content": "Hello, how are you?"}]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs,max_new_tokens=8000)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Hello! I'm an AI assistant, so I don't have feelings or emotions. However, thank you for asking! How can I assist you today?<|im_end|>


# Task 1.2

## 1.2.1

In [26]:
def extract_book_metadata(book_text_content,temperature,do_sample_flag=False):
  prompt_template=f"""
  You are an expert digital transformation specialist tasked with extracting book metadata.
  Your goal is to extract specific information from the provided book description and output it strictly as a JSON object.
  Ensure the JSON is perfectly valid and can be directly loaded by a JSON parser (e.g., `json.loads()` in Python).

  The JSON object must contain only the following keys with their specified data types:
  - "title": (string) - The full title of the book.
  - "author": (string) - The name of the author.
  - "year_of_publish": (integer) - The four-digit year of publication.
  - "publisher": (string) - The name of the publisher.
  - "summary": (string) - A concise summary of the book's plot or main themes, derived directly from the text.
  - "keywords": (array of strings) - A list of descriptive keywords or genres related to the book.
  - "rating": (float) - The numerical rating of the book. If not explicitly found, use 0.0 or null.

  Example of expected JSON format:
  ```json
  {{
    "title": "Example Title",
    "author": "Example Author",
    "year_of_publish": 2023,
    "publisher": "Example Publisher",
    "summary": "This is an example summary of the book.",
    "keywords": ["example", "keyword", "genre"],
    "rating": 4.5
  }}
  ```

  ---
  Book Information to Extract Metadata From:
  {book_text_content}
  ---

  Your JSON Metadata Output:
  ```json
  """
  
  messages = [{"role": "user", "content": prompt_template}]
  text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
  model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
  outputs = model.generate(**model_inputs,max_new_tokens=8000,
                           temperature=temperature,do_sample=do_sample_flag,)
  json_string = tokenizer.decode(outputs[0][model_inputs["input_ids"].shape[-1]:],skip_special_tokens=True)
  
  clean_json_string = json_string.strip()
  if clean_json_string.startswith("```json") and clean_json_string.endswith("```"):
      clean_json_string = clean_json_string[len("```json"): -len("```")].strip()
  
  return clean_json_string  

In [ ]:
for i in range(1,8):
  with open(f"supplementary_files/test_{i}.txt") as f:
    input_text = f.read()
    json_string=extract_book_metadata(input_text,0.0)
    try:
      parsed_json = json.loads(json_string)
      print("\n--- Parsed JSON Data ---")
      print(json.dumps(parsed_json, indent=2))
      print("\nJSON successfully parsed!")
    except Exception as e:
      print(f"Could not load json: {e}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Parsed JSON Data ---
{
  "title": "BRIGHTER THAN SILENCE",
  "author": "M. Ilyas",
  "year_of_publish": 2022,
  "publisher": "glass & field",
  "summary": "A boy who won\u2019t speak. A town that won\u2019t listen. When Ayaz arrives in Riverton, he doesn\u2019t talk \u2014 but his presence says everything. Told in fragments and whispers, this novel explores grief, perception, and the stories that never get told aloud.",
  "keywords": [
    "quiet drama",
    "small-town lens",
    "grief-lit",
    "introspective"
  ],
  "rating": 4.8
}

JSON successfully parsed!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Parsed JSON Data ---
{
  "title": "Walking Without Shoes",
  "author": "L. M. Ortega",
  "year_of_publish": 2018,
  "publisher": "Harbor South Press",
  "summary": "This is a life story stitched together by silence, borderlines, and the weight of memory, narrated through letters, subway transfers, and kitchen notes.",
  "keywords": [
    "border memoir",
    "immigrant lens",
    "poetic nonfiction",
    "family fragments"
  ],
  "rating": 4.5
}

JSON successfully parsed!


...


--- Parsed JSON Data ---
{
  "title": "THE THREADS WE LEFT",
  "author": "Anika Rahman",
  "year_of_publish": 2016,
  "publisher": "olive vine house",
  "summary": "This is a story stitched across continents, through war and exile, in the quiet ways women pass on truth.",
  "keywords": [
    "intergenerational drama",
    "exile-lit",
    "memory fiction",
    "silk-and-paper stories"
  ],
  "rating": 4.7
}

JSON successfully parsed!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Parsed JSON Data ---
{
  "title": "The Quiet Collapse",
  "author": "T. Osei",
  "year_of_publish": null,
  "publisher": "moonblack editions",
  "summary": "Araba lives in the drainage canals, mapping what's left of the city with chalk. She has to choose between a dying world and one more lie.",
  "keywords": [
    "post-flood era",
    "found sound fiction",
    "survival maps",
    "quiet collapse"
  ],
  "rating": 4.5
}

JSON successfully parsed!


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



--- Parsed JSON Data ---
{
  "title": "LIKE PAPER BIRDS",
  "author": "N.",
  "year_of_publish": null,
  "publisher": "sunhill print",
  "summary": "This is an example summary of the book. It talks about Mina\u2019s habit of leaving small paper cranes on trains after a caf\u00e9 fire and an unexpected guest, emphasizing kindness and love.",
  "keywords": [
    "soft-lit drama",
    "burnt edges",
    "train station hearts"
  ],
  "rating": 4.1
}

JSON successfully parsed!

--- Parsed JSON Data ---
{
  "title": "Every Satellite Drifts",
  "author": "Unknown",
  "year_of_publish": null,
  "publisher": "Unknown",
  "summary": "This story follows a technician who can't remember if they're repairing the world or preserving its ruins, set across abandoned skyports and fractured timelines.",
  "keywords": [
    "satellite",
    "technician",
    "abandoned",
    "skyport",
    "memory",
    "drift",
    "ruins",
    "preservation"
  ],
  "rating": null
}

JSON successfully parsed!


Model is following the instruction well and return the info correctly in json format. When information are not mention in the text like year of publish, publisher and rating the model can follow the instruction and return null value for that particular field.

## 1.2.2

- No comment on the model's response.
- Prompt engineering required very detail instruction or info so that it can perform wells. If giving some instruction that are not clear or specific it may return something that are not expected. It takes time to enhance the prompt and also word that choose.

# Task 1.3

In [27]:
for i in range(1,8):
  with open(f"supplementary_files/test_{i}.txt") as f:
    input_text = f.read()
    json_string=extract_book_metadata(input_text,0.5,True)
    try:
      parsed_json = json.loads(json_string)
      print("\n--- Parsed JSON Data ---")
      print(json.dumps(parsed_json, indent=2))
      print("\nJSON successfully parsed!")
    except Exception as e:
      print(f"Could not load json: {e}")


--- Parsed JSON Data ---
{
  "title": "BRIGHTER THAN SILENCE",
  "author": "M. Ilyas",
  "year_of_publish": 2022,
  "publisher": "glass & field",
  "summary": "A boy who won\u2019t speak. A town that won\u2019t listen. This novel explores grief, perception, and the stories that never get told aloud.",
  "keywords": [
    "quiet drama",
    "small-town lens",
    "grief-lit",
    "introspective"
  ],
  "rating": 4.8
}

JSON successfully parsed!

--- Parsed JSON Data ---
{
  "title": "Walking Without Shoes",
  "author": "L. M. Ortega",
  "year_of_publish": 2018,
  "publisher": "Harbor South Press",
  "summary": "This is a life story that spans from El Paso to New York, stitched together by silence, borderlines, and memories. It includes letters, subway transfers, and kitchen notes.",
  "keywords": [
    "border memoir",
    "immigrant lens",
    "poetic nonfiction",
    "family fragments"
  ],
  "rating": 4.5
}

JSON successfully parsed!

--- Parsed JSON Data ---
{
  "title": "FIVE DAYS

In [28]:
for i in range(1,8):
  with open(f"supplementary_files/test_{i}.txt") as f:
    input_text = f.read()
    json_string=extract_book_metadata(input_text,1.0,True)
    try:
      parsed_json = json.loads(json_string)
      print("\n--- Parsed JSON Data ---")
      print(json.dumps(parsed_json, indent=2))
      print("\nJSON successfully parsed!")
    except Exception as e:
      print(f"Could not load json: {e}")


--- Parsed JSON Data ---
{
  "title": "BRIGHTER THAN SILENCE",
  "author": "M. Ilyas",
  "year_of_publish": 2022,
  "publisher": "glass & field",
  "summary": "A boy who won\u2019t speak. A town that won\u2019t listen. This novel explores grief, perception, and the stories that never get told aloud.",
  "keywords": [
    "quiet drama",
    "small-town lens",
    "grief-lit",
    "introspective"
  ],
  "rating": 4.8
}

JSON successfully parsed!

--- Parsed JSON Data ---
{
  "title": "Walking Without Shoes",
  "author": "L. M. Ortega",
  "year_of_publish": 2018,
  "publisher": "Harbor South Press",
  "summary": "A life described through letters, subway transfers, and kitchen notes; a journey from El Paso to a rooftop in New York, stitched together by silence, borderlines, and memory.",
  "keywords": [
    "border memoir",
    "immigrant lens",
    "poetic nonfiction",
    "family fragments"
  ],
  "rating": 4.5
}

JSON successfully parsed!

--- Parsed JSON Data ---
{
  "title": "FIVE DA

Observation:
- Main difference on the summary and keywords